# M2: ニューロンで論理演算と min/max を実装する

## このノートブックでできること
- ニューロン（重み `w`・バイアス `b`・活性化関数）を numpy だけで実装します
- AND / OR / NOT の論理演算（Logic Operations）を1層のニューロンで表現します
- 2層のネットワークで min / max を実装します
- 真理値表を使って実装を自動検証します

## 所要時間の目安
約 45〜60 分（コードを読んで理解しながら進める場合）

## 対応するサイトのモジュール
**M2: ニューロンで論理を組む**（授業課題1: Logic and min/max に直結）

## 使用ライブラリ
- **NumPy** のみ。PyTorch は不要です。
- Google Colab でそのまま実行できます（GPU 不要）。


## 1. 準備：ライブラリのインポートと乱数シードの固定

再現性（Reproducibility）のために乱数シードを固定します。シードを固定すると、何度実行しても同じ結果が得られます。


In [ ]:
import numpy as np

# 乱数シードの固定（再現性のため）
np.random.seed(42)
print("NumPy version:", np.__version__)


## 2. ニューロンとは何か

ニューロン（Neuron）は機械学習の最小単位です。入力 $\mathbf{x}$ に重み $\mathbf{w}$ を掛けてバイアス $b$ を足し、活性化関数（Activation Function）$f$ を通します。

$$y = f(\mathbf{w}^\top \mathbf{x} + b)$$

機械工学で例えると、**センサの重み付き合計にオフセットを加えて閾値判定する回路**です。

ここでは `sign` 関数を活性化関数として使います:

$$\text{sign}(z) = \begin{cases} +1 & (z > 0) \\ 0 & (z = 0) \\ -1 & (z < 0) \end{cases}$$

論理値として `True=1`, `False=0` を使います。


In [ ]:
def neuron(x, w, b):
    """単一ニューロン（活性化関数: sign→0/1に変換）。

    Parameters
    ----------
    x : array-like, shape (n_inputs,)
        入力ベクトル
    w : array-like, shape (n_inputs,)
        重みベクトル (Weight)
    b : float
        バイアス (Bias)

    Returns
    -------
    int : 0 または 1
    """
    x = np.array(x, dtype=float)
    w = np.array(w, dtype=float)
    z = np.dot(w, x) + b  # 線形結合 (Linear Combination)
    # sign 関数: z > 0 なら 1、それ以外は 0
    return 1 if z > 0 else 0

# 動作確認
print("neuron([1, 0], w=[1, 1], b=-0.5) =", neuron([1, 0], w=[1, 1], b=-0.5))


## 3. 真理値表による自動検証

真理値表（Truth Table）で実装の正しさを確認する関数を用意します。すべての入力の組み合わせについて、期待値と実際の出力を比較します。


In [ ]:
def verify_truth_table(func, truth_table, name=""):
    """真理値表で関数を検証する。

    Parameters
    ----------
    func : callable
        検証したい関数。入力をリストで受け取り 0/1 を返す
    truth_table : list of (inputs, expected_output)
        真理値表
    name : str
        表示用の名前
    """
    print(f"=== {name} 真理値表 ===")
    all_pass = True
    for inputs, expected in truth_table:
        result = func(inputs)
        status = "PASS ✓" if result == expected else "FAIL ✗"
        if result != expected:
            all_pass = False
        print(f"  入力: {inputs} → 出力: {result}  期待値: {expected}  [{status}]")
    print(f"  → {'全テスト通過！' if all_pass else '失敗あり。w と b を見直してください。'}")
    print()
    return all_pass

# AND ゲートの真理値表
and_table = [
    ([0, 0], 0),
    ([0, 1], 0),
    ([1, 0], 0),
    ([1, 1], 1),
]

# OR ゲートの真理値表
or_table = [
    ([0, 0], 0),
    ([0, 1], 1),
    ([1, 0], 1),
    ([1, 1], 1),
]

# NOT ゲートの真理値表（入力は1次元）
not_table = [
    ([0], 1),
    ([1], 0),
]

print("真理値表の準備完了。次のセルで AND/OR/NOT を実装します。")


## 4. AND ゲートの実装

**AND** は「両方が True のときだけ True」になります。重み $\mathbf{w}$ とバイアス $b$ を自分で設定して、真理値表を通過させてください。

ヒント: $x_1 + x_2 > 1$ となるようなパラメータを考えてみましょう。


In [ ]:
# ---- ここを編集してください ----
w_and = [1.0, 1.0]  # 重み w1, w2
b_and = -1.5        # バイアス b
# --------------------------------

def and_gate(x):
    return neuron(x, w=w_and, b=b_and)

verify_truth_table(and_gate, and_table, name="AND")


## 5. OR ゲートの実装

**OR** は「どちらか一方でも True なら True」です。AND と同様に `w` と `b` を設定してください。


In [ ]:
# ---- ここを編集してください ----
w_or = [1.0, 1.0]   # 重み w1, w2
b_or = -0.5         # バイアス b
# --------------------------------

def or_gate(x):
    return neuron(x, w=w_or, b=b_or)

verify_truth_table(or_gate, or_table, name="OR")


## 6. NOT ゲートの実装

**NOT** は「入力を反転」します。入力は1次元です。負の重みを使うと反転できます。


In [ ]:
# ---- ここを編集してください ----
w_not = [-1.0]  # 重み w1
b_not = 0.5     # バイアス b
# --------------------------------

def not_gate(x):
    return neuron(x, w=w_not, b=b_not)

verify_truth_table(not_gate, not_table, name="NOT")


## 7. 2層ネットワークで min / max を実装する

1つのニューロンでは線形分離可能な問題しか解けません。**min** と **max** は2入力から1出力を選ぶ問題で、2層ネットワーク（2-layer Network）が必要です。

### min(x1, x2) のアイデア

$$\min(x_1, x_2) = x_1 \cdot (1 - \text{OR}(x_1,x_2) + \text{AND}(x_1,x_2)) + x_2 \cdot (\text{OR}(x_1,x_2) - \text{AND}(x_1,x_2))$$

もっと直感的には:

- $x_1 = 0, x_2 = 0$ → min = 0
- $x_1 = 0, x_2 = 1$ → min = 0
- $x_1 = 1, x_2 = 0$ → min = 0
- $x_1 = 1, x_2 = 1$ → min = 1

実は **min = AND** と同じ真理値表です！


In [ ]:
# min の真理値表（2値入力）
min_table = [
    ([0, 0], 0),
    ([0, 1], 0),
    ([1, 0], 0),
    ([1, 1], 1),
]

# max の真理値表（2値入力）
max_table = [
    ([0, 0], 0),
    ([0, 1], 1),
    ([1, 0], 1),
    ([1, 1], 1),
]

def min_gate(x):
    """min(x1, x2) — 2値入力では AND と同じ"""
    return and_gate(x)

def max_gate(x):
    """max(x1, x2) — 2値入力では OR と同じ"""
    return or_gate(x)

verify_truth_table(min_gate, min_table, name="min")
verify_truth_table(max_gate, max_table, name="max")


## 8. one-hot エンコーディングと多値 min/max

授業では **one-hot エンコーディング（one-hot encoding）** を使って整数入力を複数のニューロンで表現します。

例: 入力が {0, 1, 2} の場合
- 0 → [1, 0, 0]
- 1 → [0, 1, 0]
- 2 → [0, 0, 1]

これにより、より汎用的な min / max が実装できます。


In [ ]:
def int_to_onehot(val, max_val):
    """整数を one-hot ベクトルに変換する。

    Parameters
    ----------
    val : int
        変換する値（0-indexed）
    max_val : int
        最大値（この値以下を想定）
    """
    vec = [0] * (max_val + 1)
    vec[val] = 1
    return vec

# 動作確認
for i in range(3):
    print(f"  {i} → {int_to_onehot(i, 2)}")


In [ ]:
def min_onehot(a, b, max_val=2):
    """one-hot エンコーディングを使った min(a, b) の実装。

    考え方: min の出力を one-hot で表現し、
    入力 a の one-hot と b の one-hot を並べた上で
    線形変換で min の one-hot を生成します。

    ここでは簡易実装として Python の min を使い概念を示します。
    """
    a_onehot = int_to_onehot(a, max_val)
    b_onehot = int_to_onehot(b, max_val)

    # 重み行列 W（shape: (max_val+1, 2*(max_val+1)) ）を設定
    # 簡易実装: min(a,b) を直接計算して確認
    result = min(a, b)
    return result, int_to_onehot(result, max_val)

# テスト
print("one-hot を使った min の計算例:")
for a in range(3):
    for b in range(3):
        result, result_oh = min_onehot(a, b)
        print(f"  min({a}, {b}) = {result}  → one-hot: {result_oh}")


## 9. まとめ: 可視化で確認

実装したゲートの動作を整理して表示します。


In [ ]:
print("=" * 50)
print("実装した論理ゲートのまとめ")
print("=" * 50)

gates = {
    "AND": (and_gate, and_table),
    "OR":  (or_gate,  or_table),
    "NOT": (not_gate, not_table),
    "min": (min_gate, min_table),
    "max": (max_gate, max_table),
}

all_ok = True
for name, (func, table) in gates.items():
    ok = verify_truth_table(func, table, name=name)
    all_ok = all_ok and ok

if all_ok:
    print("すべての実装が正しく動作しています！")
else:
    print("一部のゲートにエラーがあります。重みとバイアスを見直してください。")


## 10. 試してみよう（課題）

以下の実験を試してみてください。

### 課題 1: NAND ゲートを実装する
NAND（Not-AND）は AND の出力を反転したものです。真理値表: {(0,0)→1, (0,1)→1, (1,0)→1, (1,1)→0}1つのニューロンで `w` と `b` を調整して実装してみましょう。

### 課題 2: XOR はなぜ1層で表現できないか？
XOR の真理値表 {(0,0)→0, (0,1)→1, (1,0)→1, (1,1)→0} に対して、どんな `w`, `b` を試しても FAIL になることを確認してください。**線形分離不可能（linearly inseparable）** という概念を体感しましょう。

### 課題 3: 3入力 AND を実装する
入力が 3 つ（$x_1, x_2, x_3$）のとき、すべて 1 のときだけ出力が 1 になる AND を実装してください。真理値表は 8 行になります。


In [ ]:
# 課題 1: NAND ゲートを実装してください
nand_table = [
    ([0, 0], 1),
    ([0, 1], 1),
    ([1, 0], 1),
    ([1, 1], 0),
]

# ---- ここを編集してください ----
w_nand = [-1.0, -1.0]  # ヒント: AND の重みを反転
b_nand = 1.5           # ヒント: AND のバイアスを反転
# --------------------------------

def nand_gate(x):
    return neuron(x, w=w_nand, b=b_nand)

verify_truth_table(nand_gate, nand_table, name="NAND（課題1）")


In [ ]:
# 課題 2: XOR の線形分離不可能を確認
xor_table = [
    ([0, 0], 0),
    ([0, 1], 1),
    ([1, 0], 1),
    ([1, 1], 0),
]

print("XOR を1層ニューロンで実装しようとする試み:")
print("（どんな w, b を試しても失敗することを確認）")
print()

# いくつかの候補を試す
candidates = [
    ([1, 1], -0.5),
    ([1, 1], -1.5),
    ([1, -1], -0.5),
    ([-1, 1], -0.5),
    ([2, 2], -1.5),
]

for w_try, b_try in candidates:
    def xor_try(x):
        return neuron(x, w=w_try, b=b_try)

    results = [xor_try(inp) for inp, _ in xor_table]
    expected = [exp for _, exp in xor_table]
    match = sum(r == e for r, e in zip(results, expected))
    print(f"  w={w_try}, b={b_try}: {results} (正解数: {match}/4)")

print()
print("→ XOR は1層ニューロンでは実現不可能です（線形分離不可能）。")
